## What is molecular dynamics actually doing?

### Michael Shirts, CU Boulder

First, we import some needed libraries:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

To keep it simple, let's start with a single particle moving in a 1D potential energy well. 

First, I define both the force and the potential energy.  I'll define a quartic, or 4th power well.  We do want a potential that keeps the particle localized, and not shooting off to infinity.

Note, the potential and force have to agree with each other, or the physics won't work!  Specifically, $F = -\frac{dU}{dx}$. 

In [ ]:
def quadratic_potential(x):
    pot = x**4
    return pot

def quadratic_force(x):
    #the force is -dU/dx
    f = -4*x**3  
    return f

Let's plot position versus energy so we understand the potential we are moving in.

In [ ]:
xvar = np.linspace(-4,4,1000)
plt.plot(xvar,quadratic_potential(xvar))
plt.xlabel('position')
plt.ylabel('energy')
plt.title('energy versus position')
plt.show()

In [ ]:
xvar = np.linspace(-4,4,1000)
plt.plot(xvar,quadratic_force(xvar))
plt.xlabel('position')
plt.ylabel('force')
plt.title('force versus position')
plt.show()

We usually specify units in a graph, but we haven't put any units into the equation, so the $x$ and $y$ axes are unitless at this point. 

Let's review Newton's equation of motion ($F=ma$), the differential equation we are trying to solve:

$$ \frac{\partial^2 x}{\partial t^2} = a = \frac{1}{m}F(x) = -\frac{1}{m}\frac{\partial U(x)}{\partial x} $$
As we saw, the most direct approximation to solving this is using the Taylor series expansions in both $x$ and $v$. We're keeping it simple to start and just using the terms linearly proportional to $\Delta t$.
$$
x(t+\Delta t) \approx x(t) + \frac{dx}{dt}\bigg{|}_{t}\Delta t + \mathcal{O}(\Delta t^2) 
$$
$$
x(t+\Delta t) \approx x(t) + v(t)\Delta t + \mathcal{O}(\Delta t^2) 
$$
And for the Talyor series for velocity, we get:
$$
v(t + \Delta t) \approx v(t) + \frac{dv}{dt}\bigg{|}_{t}\Delta t + \mathcal{O}(\Delta t^2)
$$
$$
v(t+ \Delta t) \approx v(t) + \frac{1}{m} F(x) \Delta t + \mathcal{O}(\Delta t^2)  
$$
These equations only involves quantities we can get out of our simulation - our position, our velocity, and the forces.  We don't have any physical quantities involved with higher power derivatives of the position.

This is accurate to $\mathcal{O}(\Delta t^2)$, which means that as $\Delta t$ decreases, the errors in $x$ and $v$ at each step decrease proportional to $\Delta t^2$.  The algorithm is generally called Euler integration.  

More precisely, each step is accurate to $\mathcal{O}(\Delta t^2)$, but since we need to add together $\tau/\Delta t$ steps to simulate out to time $\tau$, the overall error scales as $\mathcal{O}(\Delta t)$

In [ ]:
def euler(x,v,force,m=1,dt=1):

    '''
    a function that executes one step of numerical integration
    inputs:
        x (current position)
        v (current velocity)
        f (a function that computes the force)
        m = mass of particle
        dt = time interval        
    '''
    
    fx = force(x)  

    x += v*dt
    v += dt*fx/m
    
    return x,v

Now let's repeat this step many, _many_, times, and keep track of the results.

In [ ]:
def do_some_md(nsteps=100, m=1, dt=0.1, init_x=0, init_v=0, 
               algorithm=euler, force=quadratic_force, potential=quadratic_potential):
    '''
    inputs: 
        nsteps = number of steps to take
        m = mass of the particle
        dt = timestep
        init_x = initial value of x
        init_v = initial value of v
        algorithm =  the algorithm to use (just one for now)
        
    outputs: 
        a dictionary of the results over the nsteps
    '''
    # we want to store the trajectories
    xs = np.zeros(nsteps)
    vs = np.zeros(nsteps)

    x = init_x
    v = init_v

    # store the initial positions
    xs[0] = x
    vs[0] = v
    
    # besides the trajectories, we are also interested 
    # in the energy of the particle - potential, kinetic, and total energy - 
    # as Newton's equations of motion conserve energy
    
    PEs = np.zeros(nsteps)
    KEs = np.zeros(nsteps)

    KEs[0] = 0.5*m*(v**2)  # kinetic energy = 1/2 mv^2
    PEs[0] = potential(x)

    # take a number of MD steps
    # the first step is the zeros above

    for i in range(1,nsteps):
        
        xnew,vnew = algorithm(x,v,force,m=m,dt=dt)
                        
        x = xnew
        v = vnew

        # store the current points in the trajectory
        xs[i] = x
        vs[i] = v

        # calculate kinetic and potential
        KEs[i] = 0.5*m*(v**2) 
        PEs[i] = potential(x)

    # package up all of the results
    results = dict()
    results['x'] = xs
    results['v'] = vs
    results['KE'] = KEs
    results['PE'] = PEs
    results['Total_E'] = PEs + KEs # compute the total energy at each step
    
    return results

Now we are all set to run the simulation, and look at the results!

In [ ]:
nsteps = 1000
dt = 0.01
results = do_some_md(nsteps=nsteps, dt=dt,init_x=0, init_v=5, algorithm=euler)

Now, let's look at the trajectory, but FIRST - what do we expect to see?

Whenever running any sort of experiment, it's important to predict what you are going to see.  If you see something other than what you expected, you've learned something: either there is a bug in your code, or you misunderstood the problem.  If it agrees with what you expect, you have strengthened your understanding of the subject.

If you don't first predict what you are going to see, then you aren't really learning anything when you do the experiment. 
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>


In [ ]:
ts = dt*np.arange(nsteps)
plt.plot(ts,results['x'],label='x')
plt.legend()
plt.title('position versus time')
plt.show()
plt.title('velocity versus time')
plt.plot(ts,results['v'],label='v')
plt.legend()
plt.show()

It's oscillating back and forth, though not quite sinusoidal, since an $x^4$ potential gets pretty steep at the edges.  

Though . . . it looks like the particle is picking up steam a bit more each time, which doesn't sound quite right. That is not like any physics we know!  

</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>

## Diagnosing and improving our simulation

Visualizing a system is a really good way to understand what is going on. You have to look at the numbers and look at the trajectory as well.  Let's run a bit of an animation to see what is happening.

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML
%matplotlib inline

maxlen = 250
sizescale = 1.4

# downsample so we have maximum maxlen frames
x = results['x'][::int(nsteps/maxlen)]
y = results['PE'][::int(nsteps/maxlen)]
t = ts[::int(nsteps/maxlen)]

fig, ax = plt.subplots(figsize=(6, 3), dpi=80)
# adaptively find the max and min
# Dynamic limits based on your data so everything stays in frame
firstpart = int(0.4*maxlen)
minplot = sizescale*min(x[:firstpart])
maxplot = sizescale*max(x[:firstpart])
xplot = np.linspace(minplot,maxplot,100)
ax.plot(xplot, quadratic_potential(xplot), color='royalblue', lw=2.5, label='Potential Energy $U(x)$')
ax.set_xlabel('X Position')
ax.set_xlim(minplot, maxplot )
ax.set_ylim(-1, sizescale*max(y[:firstpart]) + 1)
ax.set_title('Potential Energy versus Position')
ax.grid(True, axis='x', linestyle='--', alpha=0.4)
fig.tight_layout()
ax.legend(loc='upper center')

point, = ax.plot([], [], 'ro', ms=12, label='Object') 
time_text = ax.text(0.38, 0.5, '', transform=ax.transAxes, fontfamily='monospace',
                    fontsize=10, bbox=dict(facecolor='white', alpha=0.6))

def init():
    """Initializes the background of the animation."""
    point.set_data([], [])
    time_text.set_text('')
    return point, time_text

def update(frame):
    """Updates the position of the point for each frame."""
    current_x = x[frame]
    current_y = y[frame]
    current_t = t[frame]
    point.set_data([current_x], [current_y])
    
    # Update the onscreen data readout
    time_text.set_text(
        f'{'Time:':12s}{current_t:6.2f}\n'
        f'{'Position(x):':12s}{current_x:6.2f}\n'
        f'{'Pot. E. (U):':12s}{current_y:6.2f}'
    )
    return point, time_text

# interval is in ms per frame
ani = animation.FuncAnimation(
    fig, update, frames=len(t), init_func=init, blit=True, interval=50, repeat=True
)
plt.close(fig) 
HTML(ani.to_jshtml())

We have seen that it's not behaving correctly.  What quantitative information can we look at to diagnose it?

## Energy conservation and improving our approach

So, we know that the particle is picking up steam as it goes along. We know that Newton's equations of motion should conserve total energy.  Let's plot the potential, kinetic, and total energies as a function of time and see what they look like.

In [ ]:
plt.title('energy versus time')
plt.plot(ts, results['PE'], label="potential")
plt.plot(ts, results['KE'], label="kinetic")
plt.plot(ts, results['Total_E'], label='total')
plt.legend()
plt.show()

So, _definitely_ not conservation of energy; energy is very much not constant. 

We had an algorithm to approximate the solutions of the differential equations of motion.  What approximations did we make?  Under what conditions did we expect it to work?  
</br>
</br>
</br>

The answer (the simplest one, at least) is that the error of the algorithm is proportional to $\Delta t^2$, the timestep.  So as the timestep gets longer, the solution should get worse.  Let's go back and do everything with a shorter timestep.  

Note that if we shorten the timestep by 10x, then we'll need to run for 10x the number of steps, so that the total time of solution is the same. 

You can see why computers are _very_ useful for this sort of modeling of differential equations.  Can you imagine doing thousands of steps by hand?!

**Exploration**: Take some time and see what happens to the trajectory and energies as you change:

1. Mass
2. Time step
3. Initial positions or velocities 
4. The potential energy function

## Improved integration algorithms
### Including more terms in the Taylor series

Let's try to make this more accurate, by going out further in the Taylor series expansion:


\begin{align} 
x(t+\Delta t) &= x(t) + \frac{dx}{dt}\bigg{|}_{t}\Delta t + \frac{1}{2} \frac{d^2x}{dt^2}\bigg{|}_{t} \Delta t^2 + \mathcal{O}(\Delta t^3) \\
             &= x(t) + v(t)\Delta t + \frac{1}{2m} F(x) \Delta t^2 + \mathcal{O}(\Delta t^3)
\end{align}

This again only involves quantities we know - our position, our velocity, and the forces.  We don't have any physical quantities involved with higher power derivatives of the position.

However, for velocity, we can still only go out two orders in $t$.  We don't have any information about $\frac{da}{dt} = \frac{d^2v}{dt^2} = \frac{d^3x}{dt^3}$ we can easily extract from our simulation.

\begin{align} 
v(t+\Delta t) &= v(t) + \frac{dv}{dt}\bigg{|}_{t}\Delta t + \frac{1}{2}\frac{d^2v}{dt^2}\bigg{|}_{t}\Delta t^2 +  \mathcal{O}(\Delta t^3)\\
&= v(t) + \frac{1}{m} F(x) \Delta t + \mathcal{O}(\Delta t^2)  
\end{align} 

This is accurate to $\mathcal{O}(\Delta t^3)$ for $x$ and $\mathcal{O}(\Delta t^2)$ for $v$ for each step, with overall accuracy simulated for a give n $\tau$ $\mathcal{O}(\Delta t^2)$ for $x$ and $\mathcal{O}(\Delta t)$ for $v$.  Let's see if this algorithm is any better than the first pass.

In [ ]:
def taylor2(x, v, force, m=1, dt=1):

    '''
    a function that executes one step of numerical integration with an increased power in the Taylor series.
    inputs:
        x (current position)
        v (current velocity)
        f (a function that computes the force)
        m = mass of particle
        dt = time interval        
    '''
    
    fx = force(x)  

    # saving the force as fx, I only need to call the force once; 
    # calling the force is the most expensive thing in simulations

    x += v*dt+0.5*(fx/m)*dt**2
    v += dt*fx/m

    return x,v

Let's compare the methods now:

In [ ]:
nsteps = 1000
dt = 0.01

results_euler = do_some_md(nsteps=nsteps, dt=dt, init_x=0, init_v=5, algorithm=euler)
results_taylor2 = do_some_md(nsteps=nsteps, dt=dt, init_x=0, init_v=5, algorithm=taylor2)

ts = dt*np.arange(nsteps)
ts = dt*np.arange(nsteps)
properties = ['x','v','PE','KE','Total_E']
for property in properties:
    plt.plot(ts, results_euler[property], label='euler')
    plt.plot(ts, results_taylor2[property], label='taylor2')
    plt.legend()
    plt.xlabel('time')
    plt.ylabel(f'{property}')
    plt.title(f'{property} vs. time')
    plt.show()

Definitely better!
</br>
</br>
</br>
</br>
</br>
</br>

### Work smarter, not harder: velocity Verlet

Let's try another algorithm, called velocity Verlet. The Verlet series of methods was developed by French physicist Loup Verlet, in 1967 - this particular variation was developed by Bill Swope in 1992 (who I collaborated with on one of my first papers as a graduate student!). 

In [ ]:
def velocity_verlet(x, v, force, m=1, dt=1):

    '''
    a function that executes one step of velocity verley numerical integration
    inputs:
        x (current position)
        v (current velocity)
        f (a function that computes the force)
        m = mass of particle
        dt = time interval        
    '''
    
    # take a half-step of velocity
    v_h = v + 0.5*dt*force(x)/m
    # full step of position
    x += v_h*dt
    # half-step of velocity
    v = v_h + 0.5*dt*force(x)/m  # this is the force at the x + delta t
    return x,v

In this algorithm, when we calculate the force at $t$, we use this force to find the velocity at $v(t + \Delta t/2)$. We then calculate the change in position between $t$ and $t+\Delta t$ using this *halfway* velocity. Then we call the force at the _new_ position to finish the step. So the velocity comes half from the force at $t$, and half from the force at $t+\Delta t$, so we're using an averaged velocity, not just the initial velocity.  

Not only that, we are also calculating the change in $x$ between $x(t)$ and $x(t+\Delta t)$ using the average velocity over that time period. 

It's not at **all** immediately obvious, but it turns out because of this averaging, the algorithm is accurate to order $\mathcal{O}(\Delta t^3)$ in _both_ $x$ and $v$, even though we only use first derivative properties (i.e. we use only the velocity to update the position, and only the force to update the velocity), and over a fixed time period $\tau$, accurate to $\mathcal{O}(\Delta t^2)$ in both $x$ and $v$.

In [ ]:
nsteps = 1000
dt = 0.01

results_euler = do_some_md(nsteps=nsteps, dt=dt, init_x=0, init_v=5, algorithm=euler)
results_taylor2 = do_some_md(nsteps=nsteps, dt=dt, init_x=0, init_v=5, algorithm=taylor2)
results_vv = do_some_md(nsteps=nsteps, dt=dt, init_x=0, init_v=5, algorithm=velocity_verlet)

ts = dt*np.arange(nsteps)
properties = ['x','v','PE','KE','Total_E']
for property in properties:
    plt.plot(ts, results_euler[property], label='euler')
    plt.plot(ts, results_taylor2[property], label='taylor2')
    plt.plot(ts, results_vv[property], label='vv')
    plt.legend()
    plt.xlabel('time')
    plt.ylabel(f'{property}')
    plt.title(f'{property} vs. time')
    plt.show()

Impressive energy conservation! 

How much does it change as we shorten the timestep?

In [ ]:
nsteps = [1000,10000,100000]
dt = [0.01,0.001,0.0001]
results = dict()
for n,d in zip(nsteps,dt):
    results[d] = do_some_md(nsteps=n, dt=d, init_x=0, init_v=5, algorithm=velocity_verlet)

properties = ['x','v','PE','KE','Total_E']
for property in properties:
    for n,d in zip(nsteps,dt):
        ts = d*np.arange(n)
        plt.plot(ts, results[d][property], label=f'dt={d}')
    plt.xlabel('time')
    plt.ylabel(f'{property}')
    plt.title(f'{property} vs. time')
    plt.legend()
    plt.show()

**Exploration**: How large a timestep _can_ we use for velocity Verlet for?
</br>
</br>
</br>
</br>
</br>

## Can we do any better? Higher order algorithms

Can we come up with algorithms that are even better, with more terms in the Taylor series expansion, or doing cleverer rearrangements of data?  It turns out people have come up with a lot of higher order algorithms, though they usually aren't used that much for molecular dynamics.  We will see why in a bit.  

Let's code up the Beeman's algorithm, which uses a more accurate Taylor series expansion of the positions and velocities, and adds some correction terms.  Crucially, it uses information from 3 different positions.  You can show that you can either use higher power derivatives, **or** use information from more than one step.  Both give you higher order algorithms.

In [ ]:
def beeman(x, v, force ,m=1, dt=1, oldx=0, corrector = True):
    
    '''
    a function that executes one step of numerical integration
    inputs:
        x (current position)
        v (current velocity)
        f (a function that computes the force)
        oldx (the previous position)
        m = mass of particle
        dt = time interval        
    '''    
    
    amdt = force(oldx)/m  # acceleration at t-dt
    at = force(x)/m # acceleration at t
    x += v*dt + dt**2*((4*at-amdt)/6)
    apdt = force(x)/m # acceleration at t+dt
    if corrector:
            v += dt*(5*apdt+8*at-amdt)/12  # Beeman's with 2nd order Adams-Moulton correction 
    else:
            v += dt*(2*apdt+5*at-amdt)/6   # Beeman's without 2nd order Adams-Moulton correction 
    return x,v

Note how Beeman's algorithm uses the force at $t-\Delta t$, $t$, and $t+\Delta t$.

We'll have to write a new `do_some_md` version to handle the extra bookkeeping for that extra step.

In [ ]:
def do_some_md_v2(nsteps=100, m=1, dt=0.1, init_x=0, init_v=0, 
                  algorithm=euler, force=quadratic_force, potential=quadratic_potential):

    '''
    inputs: 
        nsteps = number of steps to take
        m = mass of the particle
        dt = timestep
        init_x = initial value of x
        init_v = initial value of v
        algorithm =  the algorithm to use (just one for now)
        oldx = the previous position; we will need this for some algorithms later
        
    outputs: 
        a dictionary of the results over the nsteps
        
    '''
    # we want to store the trajectories
    xs = np.zeros(nsteps)
    vs = np.zeros(nsteps)

    x = init_x
    v = init_v

    # store the initial positions
    xs[0] = x
    vs[0] = v
    
    # besides the trajectories, we are interested 
    # in the energy of the particle - potential, kinetic, and total
    # as Newton's equations of motion conserve energy
    
    PEs = np.zeros(nsteps)
    KEs = np.zeros(nsteps)

    KEs[0] = 0.5*m*(v**2)  # kinetic energy = 1/2 mv^2
    PEs[0] = potential(x)

    if algorithm==beeman:
        # for Beeman, we need to back calculate the previous position for the first step
        # from the initial conditions
        oldx = init_x - dt*init_v-0.5*(dt**2*force(init_x)/m)

    # take a number of MD steps
    for i in range(1,nsteps):
        if algorithm==beeman:
            xnew,vnew = algorithm(x, v, force, m=m, dt=dt, oldx=oldx, corrector=True)
            oldx = x   # store oldx
        else:
            xnew,vnew = algorithm(x, v, force, m=m, dt=dt)            

        x = xnew
        v = vnew

        # calculate kinetic and potential
        KEs[i] = 0.5*m*(v**2) 
        PEs[i] = potential(x)

        # store the current points in the trajectory
        xs[i] = x
        vs[i] = v

    # package up all of the results
    results = dict()
    results['x'] = xs
    results['v'] = vs
    results['KE'] = KEs
    results['PE'] = PEs
    results['Total_E'] = PEs + KEs # compute the total energy at each step
    
    return results

Now, let's look at the results and compare to velocity verlet.  Let's first look for short times.

In [ ]:
nsteps = 200
dt = 0.005

init_x=0
init_v=5
m=1
results_vv = do_some_md_v2(nsteps=nsteps, dt=dt, init_x=init_x, init_v=init_v, algorithm=velocity_verlet)
results_b = do_some_md_v2(nsteps=nsteps, dt=dt, init_x=init_x, init_v=init_v, algorithm=beeman)

ts = dt*np.arange(nsteps)
properties = ['x','v','PE','KE','Total_E']
ps = 1
for property in properties:
    plt.plot(ps*ts[::ps], results_vv[property][::ps], label='vv')
    plt.plot(ps*ts[::ps], results_b[property][::ps], label='b')
    plt.xlabel('time')
    plt.ylabel(f'{property}')
    plt.title(f'{property} vs. time')
    plt.legend()
    plt.show()

We can see that we actually are conserving energy a little better; we stay closer to the initial energy. That's because the velocity has been corrected a bit. 

However, what happens when we run for longer time periods?  try running for 10 or 100 times as long!  That velocity correction term may supress the fluctuations, but it hurts the overall performance.


We can remove the velocity correction term, and what do we see?   It wiggles just about as much as velocity verlet (maybe a bit less) so we lose the short-time scale accuracy, but we get rid of the drift.

So despite the added complexity, we are not really doing any BETTER than the Verlet algorithms.  And that is what we generally see; it's hard to do better than velocity verlet. 
</br>
</br>

## Work smarter, not harder, part 2: leapfrog Verlet

We note that velocity Verlet still requires 2 calls of the force per timestep, and in most simulation situations, calculating the force is the most expensive thing to do.  Is there a way to get around this requirement of two force calls per step?

Hmm, the force we calculate at the END of the last step is the same as the force from FIRST part of the current step.  Can we just do some bookkeeping, and smush the second half of the velocity calculation from one iteration, and the first half of the velocity calculation from the next iteration together, so we only call the force once per call?

In [ ]:
def leapfrog_verlet(x,v,force,m=1,dt=1):
    
    '''
    a function that executes one step of numerical integration
    inputs:
        x (current position)
        v (current velocity)
        f (a function that computes the force)
        m = mass of particle
        dt = time interval        
    '''    
    x += v*dt  
    v += dt*force(x)/m   # we assume we are being passed in the velocity at t+dt/2
    
    return x,v

Then what we get at each iteration of the algorithm is the position at each timestep, and the velocity offset by 1/2 a timestep.  The velocities are leapfrogging the positions, rather than being in sync with them.

Let's try it!

In [ ]:
nsteps = 1000
dt = 0.01

results_vv = do_some_md(nsteps=nsteps, dt=dt, init_x=0, init_v=5, algorithm=velocity_verlet)
results_lv = do_some_md(nsteps=nsteps, dt=dt, init_x=0, init_v=5, algorithm=leapfrog_verlet)

properties = ['x','v','PE','KE','Total_E']
for property in properties:
    ts = dt*np.arange(nsteps)
    plt.plot(ts, results_vv[property], label='vv')
    plt.plot(ts, results_lv[property], label='lv')
    plt.xlabel('time')
    plt.ylabel(f'{property}')
    plt.title(f'{property} vs. time')
    plt.legend()
    plt.show()

Now, the trajectory looks similar, but the conserved energy is off a little.  Why? 

**Question**: Pause and think: what properties am I calculating wrong? 
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>

**Answer**: We are trying to compute the energy with the potential from the full steps, and the kinetic energy from the half steps, in between. If we try to add those together, we'll be getting a total energy that includes energies from different times, which isn't valid. We need a full step kinetic energy. 

There's a couple of options here.  Let's do the bookkeeping for the kinetic energy by calculating the true full step velocity at each point (one could also average the kinetic energies at $t+\Delta t/2$ and $t-\Delta t/2$, but we don't do that here).

In [ ]:
def do_some_md_v3(nsteps=100, m=1, dt=0.1, init_x=0, init_v=0, 
                  algorithm=euler, force=quadratic_force, potential=quadratic_potential):

    '''
    inputs: 
        nsteps = number of steps to take
        m = mass of the particle
        dt = timestep
        init_x = initial value of x
        init_v = initial value of v
        algorithm =  the algorithm to use (just one for now)
        oldx = the previous position; we will need this for some algorithms later
        
    outputs: 
        a dictionary of the results over the nsteps
        
    '''
    # we want to store the trajectories
    xs = np.zeros(nsteps)
    vs = np.zeros(nsteps)

    # store the initial positions
    x = init_x
    v = init_v
    xs[0] = x
    vs[0] = v
    
    # besides the trajectories, we are interested 
    # in the energy of the particle - potential, kinetic, and total
    # as Newton's equations of motion conserve energy
    
    PEs = np.zeros(nsteps)
    KEs = np.zeros(nsteps)
    Es = np.zeros(nsteps)

    force_init = force(init_x)
    PEs[0] = potential(x)
    if algorithm==leapfrog_verlet:
        # We need to kick it backwards a half-step to calculate the step 0 kinetic energy, 
        # since we expect the inputs are velocities at t + 0.5*dt
        v_full = init_v - (0.5*dt/m)*force_init
        KEs[0] = 0.5*m*(v_full**2)  # step 0 velocities
    else:
        KEs[0] = 0.5*m*(v**2)  # step 0 velocities - this is already the full step velocity

    if algorithm==beeman:
        # for Beeman, we need to back calculate the previous position for the first step
        # from the initial conditions
        oldx = init_x - dt*init_v-0.5*(dt**2*force_init/m)

    # take a number of MD steps
    for i in range(1,nsteps):
        if algorithm==beeman:
            xnew,vnew = algorithm(x, v, force, m=m, dt=dt, oldx=oldx)
            oldx = x   # store oldx for next beeman iteration
        else:
            xnew,vnew = algorithm(x, v, force, m=m, dt=dt)

        if algorithm == leapfrog_verlet:
            # average the old (t+dt/2) and new (t+3dt/2) velocities together 
            # to get the t+dt velocities
            KEs[i] = 0.5*m*(0.5*(v+vnew))**2
        else: 
            # kinetic energy at t+dt
            KEs[i] = 0.5*m*(vnew)**2  
        PEs[i] = potential(xnew)

        # increment to the new step velocities
        x = xnew
        v = vnew

        # store the current points (t+dt) in the trajectory
        xs[i] = x
        vs[i] = v

    # package up all of the results
    results = dict()
    results['x'] = xs
    results['v'] = vs
    results['KE'] = KEs
    results['PE'] = PEs
    results['Total_E'] = PEs + KEs # compute the total energy at each step
    
    return results

In [ ]:
nsteps = 1000
dt = 0.01

results_vv = do_some_md_v3(nsteps=nsteps, dt=dt, init_x=0, init_v=6, algorithm=velocity_verlet)
results_lv = do_some_md_v3(nsteps=nsteps, dt=dt, init_x=0, init_v=6, algorithm=leapfrog_verlet)

ts = dt*np.arange(nsteps)
properties = ['x','v','PE','KE','Total_E']
for property in properties:
    ts = dt*np.arange(nsteps)
    plt.plot(ts, results_lv[property], label='lv')
    plt.plot(ts, results_vv[property], label='vv')
    plt.xlabel('time')
    plt.ylabel(f'{property}')
    plt.title(f'{property} vs. time')
    plt.legend()
    plt.show()

So we are generating the same trajectories (up to potentially some rounding), with the same positions and velocities, doing the bookkeeping a bit differently, and saving time at each step.

</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>

## More complicated potentials

Now let's try a more complicated potential energy function!

In [ ]:
# double well potential
def double_potential(x):
    pot = x**4 - 8*x**3 + 15*x**2
    return pot

def double_force(x):
    f = -(4*x**3 - 24*x**2 + 30*x)
    return f

In [ ]:
xlist = np.linspace(-2,6,1000)
plt.plot(xlist,double_potential(xlist))
plt.show()

In [ ]:
nsteps = 200
dt = 0.01
init_x = 0
init_v = 6
m=1
results_euler = do_some_md_v3(nsteps=nsteps, dt=dt, m=m, 
                                init_x=init_x, init_v=init_v, algorithm=euler,
                                potential=double_potential,force=double_force)
results_lv = do_some_md_v3(nsteps=nsteps, dt=dt, m=m, 
                        init_x=init_x, init_v=init_v, algorithm=leapfrog_verlet,
                                potential=double_potential,force=double_force)
results_beeman = do_some_md_v3(nsteps=nsteps, dt=dt, m=m, 
                            init_x=init_x, init_v=init_v, algorithm=beeman,
                                  potential=double_potential,force=double_force)

ts = dt*np.arange(nsteps)
properties = ['x','v','PE','KE','Total_E']
for property in properties:
    ts = dt*np.arange(nsteps)
    plt.plot(ts,results_euler[property],label='euler')
    plt.plot(ts,results_lv[property],label='leapfrog_verlet')
    plt.plot(ts,results_beeman[property],label='beeman')
    plt.title(f'{property} vs. time')
    plt.legend()
    plt.show()

In [ ]:
nsteps = 500
dt = 0.04

results_lv = do_some_md_v3(nsteps=nsteps,dt=0.008,init_x=-6,init_v=0,algorithm=leapfrog_verlet,
                          potential=double_potential,force=double_force)

ts = dt*np.arange(nsteps)
properties = ['x','v','PE','KE','Total_E']
for property in properties:
    ts = dt*np.arange(nsteps)
    plt.plot(ts,results_lv[property],label='leapfrog')
    plt.title(f'{property} vs. time')
    plt.legend()
    plt.show()

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML
%matplotlib inline

maxlen = 250
minplot = -1.5
maxplot = 6.5
# downsample so we have maximum maxlen frames
x = results_lv['x'][::int(nsteps/maxlen)]
y = results_lv['PE'][::int(nsteps/maxlen)]
t = ts[::int(nsteps/maxlen)]

fig, ax = plt.subplots(figsize=(6, 3), dpi=80)
# adaptively find the max and min
# Dynamic limits based on the data so everything stays in frame
xplot = np.linspace(minplot,maxplot,100)
ax.plot(xplot, double_potential(xplot), color='royalblue', lw=2.5, label='Potential Energy $U(x)$')
ax.set_xlabel('X Position')
ax.set_xlim(minplot, maxplot)
ax.set_ylim(-20, 30)
ax.set_title('Potential Energy versus Position')
ax.grid(True, axis='x', linestyle='--', alpha=0.4)
fig.tight_layout()
ax.legend(loc='upper center')

point, = ax.plot([], [], 'ro', ms=12, label='Object') 
time_text = ax.text(0.38, 0.5, '', transform=ax.transAxes, fontfamily='monospace',
                    fontsize=10, bbox=dict(facecolor='white', alpha=0.6))

def init():
    """Initializes the background of the animation."""
    point.set_data([], [])
    time_text.set_text('')
    return point, time_text

def update(frame):
    """Updates the position of the point for each frame."""
    current_x = x[frame]
    current_y = y[frame]
    current_t = t[frame]
    point.set_data([current_x], [current_y])
    
    # Update the onscreen data readout
    time_text.set_text(
        f'{'Time:':12s}{current_t:6.2f}\n'
        f'{'Position(x):':12s}{current_x:6.2f}\n'
        f'{'Pot. E. (U):':12s}{current_y:6.2f}'
    )
    return point, time_text

# interval is in ms per frame
ani = animation.FuncAnimation(
    fig, update, frames=len(t), init_func=init, blit=True, interval=50, repeat=True
)
plt.close(fig) 
HTML(ani.to_jshtml())

</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>

## Symplectic algorithms and lack of drift ##

One thing you may be noting is that when using both of the Verlet variants or the non-corrected Beeman's algorithm;  even if the total energy is wiggling around a lot, when steps are larger, the average energy doesn't really drift up and down, but it just wiggles around the same constant value. 

The Verlet algorithm is what is called a *symplectic* algorithm.  We won't go into the full definition of what symplectic means; it's rather complicated. One key property of symplectic agorithms is that they conserve phase space, even if they have discrete timesteps. This conservation of phase space is a property that the original Newton's equations of motion have. Symplectic algorithms are also time-reversible, which Newton's equations also are. 

Both Verlet variants are symplectic.  Beeman's, without the corrector, is symplectic, but with the corrector is not symplectic.  Euler is not symplectic. 

Symplectic algorithms also have something called a *shadow Hamilonian*. The Hamiltonian is just a fancy name for the function that gives us the total energy (i.e. the kinetic + potential energy).  The fact that these algorithms have a shadow Hamiltonian means that these discrete algorithms, even with a larger timestep, exactly conserve a quantity that is close to, but not exactly, the true Hamiltonian.  

Now, the bigger the timestep gets, the further the energy gets from this exactly conserved shadow energy.  With some fancy math, one can show that the difference between the shadow Hamiltonian and the true Hamiltonian (i.e. the one we get if we used infinitely small time steps), $\tilde{H} - H$, is proportional to $\Delta t^2$ for the Verlet algorithms, with higher order terms proportional to $\Delta t^3$ or higher.   That means, if $\Delta t$ is sufficiently small, the true energy (corresponding to the true Hamiltonian) will just bounce around the shadow energy, which is a constant.  But it won't drift away, either up or down.  Other types of algorithms can't do this!

Eventually, if you get a large enough timestep, then the sum of error terms diverges, which means the physical Hamiltonan we are modeling diverges from the shadow Hamiltonian, and the simulation crashes. 

</br> There _are_ symplectic algorithms that are higher order, but they require multiple calculations of the force per step.  The Verlet algorithms, if you do the bookkeeping right, require only one force evaluation per step.   

But these higher order algorithms do not scale well. Let's say we pick one that requires 2 force evaluations per step, and usually, recall, that the force calculation is them most expensive thing.  

It turns out the steps sizes that you can take with these algorithms are less than 2x as long, so the total efficiency of simulation (measured in, say, ns/hour) is not as good when you use the more complicated algorithms.

Because of this, essentially all current MD programs use either velocity Verley or leapfrog Verlet, or some other slight variant; as simple as it is, it's just the best bang for the buck. 

If you want to dig into a lot more math, see "Geometric numerical integration illustrated by the Stormer–Verlet method" E. Hairer, C. Lubich, G. Wanner, _Acta Numerica_ (2003), pp. 399–450, at http://doi.org/10.1017/S0962492902000144), but be warned . . . it's a _lot_ more math.

</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>

## Fluctuation behavior of sympletic integrators

We have seen that the fluctuations around the average get smaller as the timestep decreases for symplectic integrators, as it gets closer to that constant energy.  How does the RMSD of the total energy scale with timestep, given a fixed total simulation time?  


To answer this, we run a number of simulations for the same amount of time, with the timestep decreasing by a factor of 2 each time (and therefore the number of timesteps doubling each time!)   

We then calculate the average of each simulation and calculate the root mean square deviation (RMSD) of the energy - which is just another name for the standard deviation of the energy from that average value. 

In [ ]:
nsteps = [1000,2000,4000,8000,16000,32000,64000,128000]
dts    = [0.08,0.04,0.02,0.01,0.005,0.0025,0.00125,0.000625]  #nsteps * dt is constant

RMSDs = list()
aves = list()
for n,dt in zip(nsteps,dts):
    results = do_some_md_v3(nsteps=n,dt=dt,init_x=-0.8,init_v=0,algorithm=leapfrog_verlet)
    aves.append(np.mean(results['Total_E'][1:]))
    RMSDs.append(np.std(results['Total_E'][1:]))
    ts = dt*np.arange(n)
    plt.plot(ts,results['Total_E'], label='total E ave')
    plt.savefig(f'energy_t_{dt}.pdf')
    plt.show()

# Now let's plot the average energy
plt.plot(dts, aves, label='aves')
plt.legend()
plt.show()


# and the RMSD of the energy 
plt.plot(dts, RMSDs, label='RMSD')
plt.legend()

plt.show()

print("RMSD of step i divided by RMSD of step i+1")
for i in range(len(dts)-1):
    print(i,RMSDs[i]/RMSDs[i+1])


**What we observe**: Each time the timestep halves, the RMSD goes down by a factor of 4, so the RMSD $\propto \Delta t^2$. The error in velocity verlet is proportional to $\Delta t^2$.  This can be a really good way to check to see if your algorithm is implemented correctly; for more information on how to do this, see PT Merz and MR Shirts. "Testing for physical validity in molecular simulations," _PLoS ONE_ 13(9): e0202764 (2018) at https://doi.org/10.1371/journal.pone.0202764

Now, try above with, instead of the `velocity_verlet` algorithm, using `leapfrog_verlet`, `euler`, and `beeman` (with and without the corrector).

</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>
</br>

## Integrator stiffness and simulation crashes

What happens when the timestep gets too big with velocity Verlet? 

In [ ]:
nsteps = [  800, 400, 200, 100, 50]
dts    = [0.025,0.05, 0.1, 0.2,0.4]
for n,dt in zip(nsteps,dts):
    ts = dt*np.arange(n)
    results = do_some_md(nsteps=n,dt=dt,init_x=0,init_v=6,algorithm=velocity_verlet)
    print("MD works OK with dt=",dt)
    plt.title(f'dt={dt}')
    plt.plot(ts,results['x'], label='x')
    plt.legend()
    plt.show()
    plt.plot(ts,results['Total_E'], label='Total E')
    plt.legend()
    plt.show()

It basically behaves just fine, until it suddenly explodes!  This is common for symplectic integrators -- they work with longer and longer timesteps, until they stop working at all, and the physical Hamiltonian diverges from the true Hamiltonian.

### A stiffer potential

Let's try the problem with a higher power, steeper, potential. 

In [ ]:
def steep_potential(x):
    pot = x**16
    return pot

def steep_force(x):
    f = -16*x**15  # -dU/dx
    return f

Let's plot both this and the quartic potential we had been using before.

In [ ]:
xvar = np.linspace(-4,4,1000)
plt.plot(xvar,quadratic_potential(xvar),label='4th power')
plt.plot(xvar,steep_potential(xvar),label='16th power')
plt.xlabel('position')
plt.ylabel('energy')
plt.title('energy versus position')
plt.ylim(-0.5,600)
plt.legend()
plt.show()

What happens now when you run with increasingly longer timesteps?

In [ ]:
nsteps = [  800, 400, 200, 100, 50]
dts    = [0.025,0.05, 0.1, 0.2,0.4]
for n,dt in zip(nsteps,dts):
    ts = dt*np.arange(n)
    results = do_some_md(nsteps=n,dt=dt,init_x=0,init_v=6,
                         algorithm=velocity_verlet,
                         potential=steep_potential,
                         force=steep_force
                        )
    print("MD works OK with dt=",dt)
    plt.title(f'dt={dt}')
    plt.plot(ts,results['x'], label='x')
    plt.legend()
    plt.show()
    plt.plot(ts,results['Total_E'], label='Total E')
    plt.legend()
    plt.show()

Note that as the potential changes more abruptly, we need to take shorter and shorter steps to perform accurate integration.  Otherwise, the simulation starts to accumulate more and more error as the potential changes by a larger amount with each timestep.

For symplectic algorithms, this change from $\Delta t$ being being basically fine, with the results just a little off, to the simulation crashing, occurs over a relatively small change in $\Delta t$. Your simulation may be going along just fine for hours at a time without crashing with a given time step, but if you increase the timestep by just a little bit, then it crashes within seconds.

Very steep potentials with large forces result in differential equations that are called *stiff*. The stiffer the differential equations, meaning, the distance over which the forces (i.e. the derivatives $dU/dx$) become big, the shorter the time steps you are able to take before the simulation fails.  This is because if you take a step that's a bit too long, you experience a bigger force than you should, which means you keep moving faster, then take a step even _bigger_ than you should, moving even faster, and this process repeats until the solution is completely wrong.

**Practice**:  Can you identify the range over which this goes from working more or less OK (no real drift even if there are fluctuations) to crashing?

It turns out for molecular models, the energies change over very short distances, as we will see later.  Additionally, the speeds of atoms and molecules are very fast, so you need _very_ short time steps, and you need to be very cautious about the time step. Remember this for the future!